# AOU-1 — Cohort definition. Phase M3 / Wave 1. Drives load_qc_cohort() from src/python/aou_ld_panel.py against the AoU v7 controlled-tier WGS MatrixTable. Emits 3 checkpointed MTs for Wave 2 dev fire.

In [ ]:
import os, sys
sys.path.insert(0, "/home/jupyter/coloc_analysis/src/python")
from aou_ld_panel import init_hail, load_qc_cohort, ANCESTRY_FIELD, KING_KINSHIP_THRESHOLD
init_hail()
print(f"WORKSPACE_BUCKET = {os.environ['WORKSPACE_BUCKET']}")
print(f"GOOGLE_PROJECT = {os.environ['GOOGLE_PROJECT']}")
print(f"WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH = {os.environ['WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH']}")

In [ ]:
# Cell 3 — Primary AFR cohort (D-M3-07 PCA-primary)
mt_afr = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=False,
)
n_afr = mt_afr.count_cols()
n_var_afr = mt_afr.count_rows()
print(f"AFR PCA cohort: {n_afr} samples, {n_var_afr} variants")
# Already checkpointed to gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc.mt by load_qc_cohort()

In [ ]:
# Cell 4 — AFR sensitivity cohort (D-M3-07 self-report Black/African American sensitivity)
mt_afr_selfid = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="afr",
    sensitivity=True,
)
n_afr_selfid = mt_afr_selfid.count_cols()
print(f"AFR PCA + self-id Black/AA cohort: {n_afr_selfid} samples (subset of AFR PCA cohort)")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc.mt

In [ ]:
# Cell 5 — EUR parity cohort (D-M3-01)
mt_eur = load_qc_cohort(
    mt_path=os.environ["WGS_ACAF_THRESHOLD_MULTI_HAIL_PATH"],
    ancestry="eur",
    sensitivity=False,
)
n_eur = mt_eur.count_cols()
print(f"EUR PCA cohort: {n_eur} samples")
# Checkpoint at gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc.mt

In [ ]:
# Cell 6 — Disjoint-cohort sanity check (RESEARCH O5)
afr_samples = mt_afr.s.collect()
eur_samples = mt_eur.s.collect()
overlap = set(afr_samples) & set(eur_samples)
assert len(overlap) == 0, f"AFR and EUR cohorts overlap by {len(overlap)} samples; investigate!"
print(f"OK: AFR and EUR cohorts disjoint ({len(afr_samples)} + {len(eur_samples)} samples)")

In [ ]:
# Cell 7 — Cohort-summary table for the validation memo
import pandas as pd
cohort_summary = pd.DataFrame({
    "cohort": ["AFR_pca", "AFR_pca_selfid", "EUR_pca"],
    "n_samples": [n_afr, n_afr_selfid, n_eur],
    "n_variants": [n_var_afr, mt_afr_selfid.count_rows(), mt_eur.count_rows()],
    "kinship_threshold": [KING_KINSHIP_THRESHOLD] * 3,
    "ancestry_field": [ANCESTRY_FIELD] * 3,
    "checkpoint_path": [
        f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/mt_afr_qc.mt",
        f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/mt_afr_pca_selfid_qc.mt",
        f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/mt_eur_qc.mt",
    ],
})
cohort_summary.to_csv("cohort_summary_m3.tsv", sep="\t", index=False)
print(cohort_summary)

## Output: 3 checkpointed MTs in workspace bucket. Mirror cohort_summary_m3.tsv to NCSU GPFS at .planning/phases/m3-aou-afr-ld-panel-build/cohort_summary_m3.tsv after Wave 2 dev fire signoff (Wave 5 close-out task).